In [2]:
import pandas as pd
import numpy as np


# 1. Càrrega de dades

In [3]:
movies_dataset = pd.read_csv("../data/movies.csv")
ratings_dataset = pd.read_csv("../data/ratings.csv")

In [4]:
movies_dataset.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
movies_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 228.5 KB


In [6]:
movies_dataset.describe()

,movieId
count,9742.000000
mean,42200.353623
std,52160.494854
min,1.000000
25%,3248.250000
50%,7300.000000
75%,76232.000000
max,193609.000000


## 1.1. One-hot encoding genres i Extracció Títol i aka

In [7]:
# Extraiem l'any i el posem en una nova columna
if 'year' not in movies_dataset.columns:
    movies_dataset['year'] = movies_dataset['title'].str.extract(r'\((\d{4})\)')
    movies_dataset['title'] = movies_dataset['title'].str.replace(r'\(\d{4}\)', '', regex=True).str.strip()

# Extraiem el títol alternatiu i el posem en una nova columna: "City of Lost Children, The (Cité des enfants perdus, La)"
if 'aka' not in movies_dataset.columns:
    movies_dataset['aka'] = movies_dataset['title'].str.extract(r'\((.*?)\)')
    movies_dataset['title'] = movies_dataset['title'].str.replace(r'\(.*?\)', '', regex=True).str.strip()

# Fem un one-hot encoding a genres
movies_dataset_one_hot = pd.concat([movies_dataset, movies_dataset['genres'].str.get_dummies(sep="|")], axis=1).drop('genres', axis=1)

In [8]:
movies_dataset_one_hot.head()

,movieId,title,year,aka,(no genres listed),Action,Adventure,Animation,Children,Comedy,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story,1995,NaN,0,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji,1995,NaN,0,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men,1995,NaN,0,0,0,0,0,1,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale,1995,NaN,0,0,0,0,0,1,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II,1995,NaN,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [9]:
#save csv 
movies_dataset_one_hot.to_csv("../data/movies_one_hot.csv", index=False)
movies_dataset = movies_dataset_one_hot

In [10]:
ratings_dataset.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [11]:
ratings_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [12]:
ratings_dataset.describe()

,userId,movieId,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


In [13]:
ratings_dataset.sort_values('timestamp')

,userId,movieId,rating,timestamp
66719,429,595,5.0,828124615
66716,429,588,5.0,828124615
66717,429,590,5.0,828124615
66718,429,592,5.0,828124615
66712,429,432,3.0,828124615
...,...,...,...,...
81475,514,187031,2.5,1537674927
81477,514,187595,3.0,1537674946
81336,514,5247,2.5,1537757040
81335,514,5246,1.5,1537757059


In [14]:
ratings_dataset['userId'].nunique()

610

# 2. Divisió train/test

In [15]:
users = ratings_dataset['userId'].unique()

train_list = []
test_list = []

for i in range(users.shape[0]):
    ratings_subset = ratings_dataset.loc[ratings_dataset['userId'] == users[i]].sort_values('timestamp')
    limit = max(1, int(ratings_subset.shape[0] * 0.8))
    
    train_list.append(ratings_subset.iloc[:limit])
    test_list.append(ratings_subset.iloc[limit:])

ratings_train_set = pd.concat(train_list, ignore_index=True)
ratings_test_set = pd.concat(test_list, ignore_index=True)

ratings_train_set.to_csv("../data/train_set.csv", index=False)
ratings_test_set.to_csv("../data/test_set.csv", index=False)

movies_train_set = movies_dataset[movies_dataset['movieId'].isin(ratings_train_set['movieId'])]
movies_test_set = movies_dataset[movies_dataset['movieId'].isin(ratings_test_set['movieId'])]

movies_train_set.to_csv("../data/movies_train_set.csv", index=False)
movies_test_set.to_csv("../data/movies_test_set.csv", index=False)

Crearem ara P i Q en base a les mides que havem de definit com a training set i test set.

In [16]:
k = 20

U = ratings_train_set['userId'].nunique()
D = movies_train_set['movieId'].nunique()
P = np.random.normal(0, 0.1, (U, k))
Q = np.random.normal(0, 0.1, (D, k))
R = P @ Q.T

b_u = np.zeros(U)
b_i = np.zeros(D)

print(P.shape, Q.shape, R.shape, b_u.shape, b_i.shape)

(610, 20) (8231, 20) (610, 8231) (610,) (8231,)


# 3. Entrenament

In [18]:
epochs = 20
lambda_ = 0.02
lr = 0.005

# 1. Creació dels mapatges d'índexs únics sobre el train
unique_users = ratings_train_set['userId'].unique()
unique_movies = ratings_train_set['movieId'].unique()

user2idx = {user_id: idx for idx, user_id in enumerate(unique_users)}
movie2idx = {movie_id: idx for idx, movie_id in enumerate(unique_movies)}

n_users = len(unique_users)
n_movies = len(unique_movies)

# 2. Inicialització de paràmetres amb les mides reals
mu = ratings_train_set['rating'].mean()
b_u = np.zeros(n_users)
b_i = np.zeros(n_movies)
P = np.random.normal(0, 0.1, (n_users, k))
Q = np.random.normal(0, 0.1, (n_movies, k))

# Extract Data Arrays per a màxim rendiment
users_arr = ratings_train_set['userId'].map(user2idx).to_numpy()
movies_arr = ratings_train_set['movieId'].map(movie2idx).to_numpy()
ratings_arr = ratings_train_set['rating'].to_numpy()

n_samples = len(ratings_arr)

# 3. Bucle d'entrenament
for epoch in range(epochs):
    # Permutació d'índexs per fer el shuffle súper ràpid
    indices = np.random.permutation(n_samples)
    
    epoch_loss = 0

    for idx in indices:
        u = users_arr[idx]
        i = movies_arr[idx]
        r_ui = ratings_arr[idx]

        # Predicció i error
        r_ui_barret = mu + b_u[u] + b_i[i] + np.dot(P[u], Q[i])
        e_ui = r_ui - r_ui_barret

        # Guardem còpia del vector original abans d'actualitzar P
        P_u_old = P[u].copy()

        # Actualització amb SGD
        b_u[u] += lr * (e_ui - lambda_ * b_u[u])
        b_i[i] += lr * (e_ui - lambda_ * b_i[i])
        P[u] += lr * (e_ui * Q[i] - lambda_ * P_u_old)
        Q[i] += lr * (e_ui * P_u_old - lambda_ * Q[i])

        # Acumular error quadràtic pur (MSE)
        epoch_loss += e_ui ** 2

    train_mse = epoch_loss / n_samples
    print(f'Epoch {epoch + 1}/{epochs} - Train MSE: {train_mse:.4f}')

Epoch 1/20 - Train MSE: 0.9232
Epoch 2/20 - Train MSE: 0.8272
Epoch 3/20 - Train MSE: 0.7912
Epoch 4/20 - Train MSE: 0.7687
Epoch 5/20 - Train MSE: 0.7520
Epoch 6/20 - Train MSE: 0.7387
Epoch 7/20 - Train MSE: 0.7274
Epoch 8/20 - Train MSE: 0.7175
Epoch 9/20 - Train MSE: 0.7083
Epoch 10/20 - Train MSE: 0.7000
Epoch 11/20 - Train MSE: 0.6917
Epoch 12/20 - Train MSE: 0.6839
Epoch 13/20 - Train MSE: 0.6760
Epoch 14/20 - Train MSE: 0.6680
Epoch 15/20 - Train MSE: 0.6598
Epoch 16/20 - Train MSE: 0.6514
Epoch 17/20 - Train MSE: 0.6426
Epoch 18/20 - Train MSE: 0.6333
Epoch 19/20 - Train MSE: 0.6236
Epoch 20/20 - Train MSE: 0.6133


# 4. Predicció